In [8]:
# ---------------------------------------------------
# Internship Recommendation System - recommend.ipynb
# ---------------------------------------------------

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ---------------------------------------------------
# 1. Load dataset
# ---------------------------------------------------
clean_path = r"D:\project hack\data\processed\internship_data_clean.csv"
filtered_path = r"D:\project hack\data\processed\internship_data_filtered.csv"

try:
    df = pd.read_csv(filtered_path)
    print(f"Filtered dataset loaded ✅ -> {df.shape}")
except FileNotFoundError:
    df = pd.read_csv(clean_path)
    print(f"Clean dataset loaded ✅ -> {df.shape}")

print("✅ Columns available:", list(df.columns))

# ---------------------------------------------------
# 2. Ensure required columns exist
# ---------------------------------------------------
if "skills_clean" not in df.columns:
    # Fallback: clean skills column if missing
    df["skills_clean"] = df["skills"].fillna("").str.lower().str.replace(r"[^a-zA-Z0-9, ]", "", regex=True)

if "combined_text" not in df.columns:
    # Create combined text (role + company + skills)
    df["combined_text"] = (
        df["role"].astype(str) + " " +
        df["company_name"].astype(str) + " " +
        df["skills_clean"].astype(str)
    )

print("✅ Final columns:", list(df.columns))

# ---------------------------------------------------
# 3. Build TF-IDF vectors
# ---------------------------------------------------
# Skills vectorizer
skills_vectorizer = TfidfVectorizer(stop_words="english")
skills_tfidf = skills_vectorizer.fit_transform(df["skills_clean"])

# Combined text vectorizer
text_vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
text_tfidf = text_vectorizer.fit_transform(df["combined_text"])

print(f"Feature shapes -> skills: {skills_tfidf.shape} text: {text_tfidf.shape}")

# ---------------------------------------------------
# 4. Recommendation function
# ---------------------------------------------------
def recommend_internships(user_skills, top_n=5, use_text=True):
    """
    Recommend internships based on user skills.
    
    Parameters:
        user_skills (str): Comma-separated skills input
        top_n (int): Number of recommendations
        use_text (bool): Use combined text or only skills
    """
    if use_text:
        user_vec = text_vectorizer.transform([user_skills])
        sims = cosine_similarity(user_vec, text_tfidf).flatten()
    else:
        user_vec = skills_vectorizer.transform([user_skills])
        sims = cosine_similarity(user_vec, skills_tfidf).flatten()
    
    # Get top recommendations
    top_idx = sims.argsort()[-top_n:][::-1]
    results = df.iloc[top_idx].copy()
    results["similarity_score"] = sims[top_idx]
    
    # Dynamic safe column selection
    cols_to_return = [
        "role",
        "company_name",
        "location" if "location" in df.columns else None,
        "stipend_numeric" if "stipend_numeric" in df.columns else "stipend",
        "duration_months" if "duration_months" in df.columns else "duration",
        "skills_clean",
        "similarity_score"
    ]
    cols_to_return = [c for c in cols_to_return if c is not None]
    
    return results[cols_to_return]

# ---------------------------------------------------
# 5. Example usage
# ---------------------------------------------------
sample_skills = "python, machine learning, data analysis"
print("\n🔎 Recommendations for:", sample_skills)
print(recommend_internships(sample_skills, top_n=5, use_text=True))


Filtered dataset loaded ✅ -> (6583, 16)
✅ Columns available: ['internship_id', 'role', 'company_name', 'location', 'duration', 'stipend', 'intern_type', 'skills', 'perks', 'hiring_since', 'opportunity_date', 'opening', 'hired_candidate', 'number_of_applications', 'website_link', 'stipend_numeric']
✅ Final columns: ['internship_id', 'role', 'company_name', 'location', 'duration', 'stipend', 'intern_type', 'skills', 'perks', 'hiring_since', 'opportunity_date', 'opening', 'hired_candidate', 'number_of_applications', 'website_link', 'stipend_numeric', 'skills_clean', 'combined_text']
Feature shapes -> skills: (6583, 492) text: (6583, 5000)

🔎 Recommendations for: python, machine learning, data analysis
                                                role             company_name  \
1227                     Machine Learning Internship  Climate Connect Digital   
6289            Machine Learning Internship (Remote)                   Avaari   
1297         Machine Learning Internship (Part ti

In [ ]:
}